# does log(log(death/birth)) track the point process params? (nested Thomas)

Same check as `loglog_death_birth_vs_params.ipynb`, run on the nested Thomas
diagrams instead of Matern. This process has 5 swept params (not 2):
`parent_intensity`, `meta_offspring`, `meta_cluster_scale` (outer/meta cluster
process) and `mean_offspring`, `cluster_scale` (inner cluster process) --
`edge_buffer` also sits in `generator_params` but it's derived, not an
independent sweep target, so it's left out.

**What this actually checks:** for each persistence diagram, take every finite
(birth, death) pair in a homology dimension and compute `log(log(death /
birth))` -- birth > 0 and death > birth always hold for this filtration (see
the sanity-check cell below, which verifies rather than assumes it), so the
ratio is always > 1 and the double log is real-valued. This is an empirical
feature-engineering probe, not derived from a theorem: it asks whether a cheap
scalar computed straight off the diagram recovers the point process's own
generative parameters, as a baseline against which the heavier persistence-image
pipeline elsewhere in this repo can be judged. A single diagram has many finite
pairs, so nine summary statistics of the per-diagram distribution of
`log(log(ratio))` are tracked -- `mean`, `median`, `min`, `max`, `std`,
`variance`, `iqr`, `skew`, `kurtosis` -- each correlated (Pearson, on
`log(param)`, since params are swept log-uniformly) against every param.

**Multi-`k`:** the DTM filtration's `k` (its density-estimation neighbourhood
size) is itself a free choice, and diagrams exist for `k in {5, 10, 15}`
(`data/nested_thomas/dtm_k{5,10,15}/diagrams.pkl`) -- this notebook now sweeps
all three instead of hardcoding `k=10`. See the "Multi-k design" note below the
imports for how it avoids turning into 27 near-duplicate figures.


In [1]:
import csv
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, skew, kurtosis

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from cloudforger.core.records import load_diagrams

OUT = ROOT / "notebooks" / "out"
OUT.mkdir(parents=True, exist_ok=True)


## Multi-k design

Nine stats x five params x three k's x two dims is 270 correlation numbers and,
if every stat got its own per-k scatter grid the way `k=10` alone used to, 27
figures. Two choices keep that from happening:

- **Compute once, across all three k, into one tidy row table** (`rows`, one
  row per `(k, diagram, dim)`), and **cache it to CSV**
  (`notebooks/out/loglog_stats_nested_thomas.csv`). Loading three
  `diagrams.pkl` files and iterating every pair is the expensive step; once
  it's done, replotting or adding a correlation view is just array indexing
  into `rows`, no rereading `diagrams.pkl`. Set `FORCE_RECOMPUTE = True` below
  to ignore the cache (needed once the in-progress `data/nested_thomas/`
  regeneration finishes and the cache should be rebuilt from the new files).
- **Two output tiers.** The full per-param scatter grids (Figures 1-9, one
  stat each, unchanged from the original single-k version) render for a single
  `DETAIL_K` only -- change it and rerun that section to see `k=5` or `k=15`
  in the same detail. The cross-k comparison instead happens in two compact
  heatmap figures (all 9 stats x 5 params x all 3 k, panel-per-k) at the end,
  which is what actually answers "does k change the story" -- see the
  discussion cell right after them.


In [2]:
PARAMS = ["parent_intensity", "meta_offspring", "meta_cluster_scale", "mean_offspring", "cluster_scale"]
K_VALUES = [5, 10, 15]
DETAIL_K = 10          # which k gets the full per-stat scatter grids (Figures 1-9)
N_SAMPLE_PER_K = 600   # diagrams sampled per k (independent RNG draw per k, same seed)
FORCE_RECOMPUTE = False  # True -> ignore the CSV cache and reread every dtm_k*/diagrams.pkl

CACHE_PATH = OUT / "loglog_stats_nested_thomas.csv"


In [3]:
# Nine summary statistics of the per-pair log(log(death/birth)) distribution,
# computed the same way for every (k, diagram, dim): mean/median/min/max locate
# it, std/variance/iqr spread it, skew/kurtosis shape it. All population
# (ddof=0 / bias=True) statistics.
STATS = {
    "mean": np.mean,
    "median": np.median,
    "min": np.min,
    "max": np.max,
    "std": np.std,
    "variance": np.var,
    "iqr": lambda a: np.percentile(a, 75) - np.percentile(a, 25),
    "skew": skew,
    "kurtosis": kurtosis,
}
ROW_FIELDS = ["k", "dim", "n_pairs"] + [f"{name}_loglog" for name in STATS] + PARAMS


In [4]:
def compute_rows_for_k(k):
    """Load dtm_k{k}/diagrams.pkl, sample N_SAMPLE_PER_K diagrams, and return one
    row per (diagram, dim) with all nine STATS of log(log(death/birth)) plus the
    diagram's generator params. Pairs with ratio <= 1 or birth <= 0 (so
    log(log(ratio)) is non-finite) are dropped individually, not just skipped at
    the diagram level -- see the sanity-check cell below for how often that fires.
    """
    diagrams, _bundle = load_diagrams(ROOT / "data" / "nested_thomas" / f"dtm_k{k}" / "diagrams.pkl")
    n_sample = min(N_SAMPLE_PER_K, len(diagrams))
    rng = np.random.default_rng(0)
    sample = [diagrams[i] for i in rng.choice(len(diagrams), size=n_sample, replace=False)]

    k_rows = []
    n_dropped = 0
    for d in sample:
        for dim in (0, 1):
            pairs = d.finite_pairs(dim)  # drops the one infinite-death H0 class
            if len(pairs) == 0:
                continue
            ratio = pairs[:, 1] / pairs[:, 0]
            with np.errstate(divide="ignore", invalid="ignore"):
                loglog = np.log(np.log(ratio))  # ratio > 1 always in practice -> log(ratio) > 0
            finite = np.isfinite(loglog)
            n_dropped += int((~finite).sum())
            loglog = loglog[finite]
            if len(loglog) == 0:
                continue
            row = {"k": k, "dim": dim, "n_pairs": len(loglog)}
            with np.errstate(invalid="ignore"):  # skew/kurtosis of a 1-pair diagram is 0/0 -> nan by design
                for name, fn in STATS.items():
                    row[f"{name}_loglog"] = float(fn(loglog))
            for p in PARAMS:
                row[p] = d.generator_params[p]
            k_rows.append(row)

    print(f"k={k}: {len(diagrams)} diagrams available, sampled {n_sample}, "
          f"dropped {n_dropped} individual pairs with ratio <= 1 or birth <= 0")
    return k_rows


if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    with open(CACHE_PATH, newline="") as f:
        rows = [
            {**{k2: int(v) if k2 in ("k", "dim", "n_pairs") else float(v) for k2, v in r.items()}}
            for r in csv.DictReader(f)
        ]
    print(f"loaded {len(rows)} rows from cache: {CACHE_PATH.relative_to(ROOT)} "
          "(set FORCE_RECOMPUTE = True above to rebuild it from diagrams.pkl)")
else:
    rows = []
    for k in K_VALUES:
        rows.extend(compute_rows_for_k(k))
    with open(CACHE_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=ROW_FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"cached {len(rows)} rows -> {CACHE_PATH.relative_to(ROOT)}")


def col(name):
    return np.array([r[name] for r in rows], dtype=float)


k_arr = col("k")
dim_arr = col("dim")
print(f"{len(rows)} total (k, diagram, dim) rows across k={K_VALUES}")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/qp252676/Desktop/point-process-tda/data/nested_thomas/dtm_k5/diagrams.pkl'

## Sanity check -- is the transform actually valid on this data?

`log(log(death/birth))` is only real-valued when `birth > 0` and `death/birth >
1`. Both were assumed (see the comments in `compute_rows_for_k` above), not
verified, in the original version of this notebook. This cell re-checks both
directly on the sampled diagrams for every k, reports how many pairs (if any)
violated them, and flags which `(k, dim)` combinations have too few finite
pairs for `std`/`skew`/`kurtosis` to mean much -- H1 in particular can have
very few pairs per diagram (occasionally 0), unlike H0 which always has at
least the 75-point floor.


In [ ]:
n_pairs_arr = col("n_pairs")

print(f"{'k':>3} {'dim':>3} {'n_diag':>7} {'pairs min':>10} {'median':>7} {'max':>6} "
      f"{'<3 pairs':>9} {'skew nan':>9} {'kurt nan':>9}")
for k in K_VALUES:
    for dim in (0, 1):
        mask = (k_arr == k) & (dim_arr == dim)
        npd = n_pairs_arr[mask]
        n_low = int((npd < 3).sum())
        n_nan_skew = int((~np.isfinite(col("skew_loglog")[mask])).sum())
        n_nan_kurt = int((~np.isfinite(col("kurtosis_loglog")[mask])).sum())
        print(f"{k:>3} {dim:>3} {mask.sum():>7} {npd.min():>10.0f} {np.median(npd):>7.0f} "
              f"{npd.max():>6.0f} {n_low:>9} {n_nan_skew:>9} {n_nan_kurt:>9}")

for p in PARAMS:
    vals = col(p)
    assert np.all(vals > 0), f"{p} has non-positive values -- log(param) would be nan/-inf"
print("all swept params strictly positive across every k -- log(param) is well-defined")

for name in STATS:
    y = col(f"{name}_loglog")
    n_nan = int((~np.isfinite(y)).sum())
    if n_nan:
        print(f"{name}_loglog: {n_nan}/{len(y)} rows are NaN across all k -- excluded from "
              "the correlations below (not treated as 0)")


## Correlation table (k=DETAIL_K in full; all k in the heatmaps below)

Printing the full `k x dim x stat x param` table would be 270 lines -- the
version below prints just `DETAIL_K` for exact numbers, and Figure 10/11
further down give the same numbers for every k as a heatmap instead.


In [ ]:
MIN_N_FOR_CORR = 10  # below this many finite (x, y) pairs, Pearson r is too noisy to report


def safe_pearsonr(x, y):
    """Pearson r over the rows where both x and y are finite, dropping NaNs
    from degenerate low-pair diagrams instead of letting them poison the whole
    coefficient (scipy.stats.pearsonr does not do this itself)."""
    mask = np.isfinite(x) & np.isfinite(y)
    n = int(mask.sum())
    if n < MIN_N_FOR_CORR:
        return np.nan, n
    r, _p = pearsonr(x[mask], y[mask])
    return r, n


print(f"k={DETAIL_K}")
print(f"{'dim':>3} {'stat':>10} {'param':>18} {'pearson r (log param)':>22} {'n':>6}")
for dim in (0, 1):
    mask = (k_arr == DETAIL_K) & (dim_arr == dim)
    for stat in STATS:
        y = col(f"{stat}_loglog")[mask]
        for param in PARAMS:
            x = np.log(col(param)[mask])
            r, n = safe_pearsonr(x, y)
            print(f"{dim:>3} {stat:>10} {param:>18} {r:>22.3f} {n:>6}")


## Figure 1 -- mean log(log(death/birth)) per diagram

Bulk behavior, sensitive to the tail (an outlier feature pulls the mean with
it, unlike the median below). `k = DETAIL_K`.


In [ ]:
def plot_grid(stat, title, fname, k=DETAIL_K):
    fig, axes = plt.subplots(2, len(PARAMS), figsize=(4 * len(PARAMS), 7), sharey="row")
    colors = {0: "tab:blue", 1: "tab:orange"}
    for row, dim in enumerate((0, 1)):
        mask = (k_arr == k) & (dim_arr == dim)
        y = col(f"{stat}_loglog")[mask]
        finite = np.isfinite(y)
        for c, param in enumerate(PARAMS):
            ax = axes[row, c]
            x = col(param)[mask]
            r, n = safe_pearsonr(np.log(x), y)
            ax.scatter(x[finite], y[finite], s=10, alpha=0.4, color=colors[dim])
            ax.set_xscale("log")
            ax.set_title(f"H{dim}, r={r:.2f}, n={n}")
            if row == 1:
                ax.set_xlabel(param)
            if c == 0:
                ax.set_ylabel(f"{stat} loglog")
    fig.suptitle(f"{title} (k={k})")
    fig.tight_layout()
    fig.savefig(OUT / fname, dpi=150)
    plt.show()


plot_grid("mean", "Mean log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "11_loglog_ratio_mean_vs_params_nested_thomas.png")


## Figure 2 -- median log(log(death/birth)) per diagram

Bulk behavior again, but robust to outlier features -- the typical pair in the
diagram rather than the tail.


In [ ]:
plot_grid("median", "Median log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "12_loglog_ratio_median_vs_params_nested_thomas.png")


## Figure 3 -- min log(log(death/birth)) per diagram

The least prominent finite feature in each diagram -- the opposite end of the
distribution from Figure 4's max.


In [ ]:
plot_grid("min", "Min log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "13_loglog_ratio_min_vs_params_nested_thomas.png")


## Figure 4 -- max log(log(death/birth)) per diagram

Most prominent feature in each diagram, against each param (log-scaled x-axis,
matching the sweep).


In [ ]:
plot_grid("max", "Max log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "14_loglog_ratio_max_vs_params_nested_thomas.png")


## Figure 5 -- std log(log(death/birth)) per diagram

How spread out the diagram's features are, in `log(log(...))` units.


In [ ]:
plot_grid("std", "Std log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "15_loglog_ratio_std_vs_params_nested_thomas.png")


## Figure 6 -- variance of log(log(death/birth)) per diagram

Same spread as Figure 5's std, but squared -- a nonlinear transform, so its
correlation with `log(param)` isn't just std's number squared and can pick out
a different relationship.


In [ ]:
plot_grid("variance", "Variance of log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "16_loglog_ratio_variance_vs_params_nested_thomas.png")


## Figure 7 -- IQR of log(log(death/birth)) per diagram

75th minus 25th percentile -- a spread measure robust to the extreme min/max
features that Figures 3-4 track directly.


In [ ]:
plot_grid("iqr", "IQR of log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "17_loglog_ratio_iqr_vs_params_nested_thomas.png")


## Figure 8 -- skew of log(log(death/birth)) per diagram

Asymmetry of the per-diagram distribution. Noisy for the low-pair-count H1
diagrams flagged in the sanity check above -- read the H1 row with that in
mind.


In [ ]:
plot_grid("skew", "Skew of log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "18_loglog_ratio_skew_vs_params_nested_thomas.png")


## Figure 9 -- kurtosis of log(log(death/birth)) per diagram

Tailedness of the per-diagram distribution (excess kurtosis, normal = 0). Same
low-pair-count caveat as skew.


In [ ]:
plot_grid("kurtosis", "Kurtosis of log(log(death/birth)) per diagram vs. process params (nested Thomas)",
           "19_loglog_ratio_kurtosis_vs_params_nested_thomas.png")


## Figure 10 / 11 -- summary: every statistic vs. every param, for every k

The cross-k comparison this notebook is actually for. Each figure is one dim
(H0, then H1), one heatmap panel per k, all nine stats x five params, signed
Pearson r annotated with the coefficient itself so color is a redundant cue,
not the only one. Read left-to-right within a row to see whether a given
stat's strongest param changes as k grows.


In [ ]:
def corr_matrix(k, dim):
    mask = (k_arr == k) & (dim_arr == dim)
    mat = np.full((len(STATS), len(PARAMS)), np.nan)
    for i, stat in enumerate(STATS):
        y = col(f"{stat}_loglog")[mask]
        for j, param in enumerate(PARAMS):
            x = np.log(col(param)[mask])
            r, _n = safe_pearsonr(x, y)
            mat[i, j] = r
    return mat


def plot_k_summary(dim, fname):
    fig, axes = plt.subplots(1, len(K_VALUES), figsize=(4.6 * len(K_VALUES), 6), sharey=True)
    im = None
    for ax, k in zip(axes, K_VALUES):
        mat = corr_matrix(k, dim)
        im = ax.imshow(mat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(len(PARAMS)))
        ax.set_xticklabels(PARAMS, rotation=45, ha="right")
        ax.set_yticks(range(len(STATS)))
        ax.set_yticklabels(list(STATS.keys()))
        ax.set_title(f"k={k}")
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat[i, j]
                label = "nan" if np.isnan(val) else f"{val:.2f}"
                color = "white" if abs(val) > 0.6 else "black"
                ax.text(j, i, label, ha="center", va="center", color=color, fontsize=8)
    fig.colorbar(im, ax=axes, shrink=0.8, label="pearson r (stat vs. log param)")
    fig.suptitle(f"H{dim}: pearson r of each log(log(death/birth)) statistic vs. "
                 "each process param, by k (nested Thomas)")
    fig.savefig(OUT / fname, dpi=150, bbox_inches="tight")
    plt.show()


plot_k_summary(0, "20_loglog_ratio_stat_summary_heatmap_H0_nested_thomas.png")


In [ ]:
plot_k_summary(1, "21_loglog_ratio_stat_summary_heatmap_H1_nested_thomas.png")


## Notes

- **Data availability:** this notebook reads `data/nested_thomas/dtm_k{5,10,15}/diagrams.pkl`.
  While those files are being (re)computed in the background (look for
  `data/nested_thomas/dtm_k*/_chunks/diagrams.group*.pkl` -- not yet merged if so), the compute cell
  above will raise `FileNotFoundError`; re-run once they land (and set `FORCE_RECOMPUTE = True` once,
  to replace any stale CSV cache built from a different snapshot). The statistics/plotting code was
  validated end-to-end against the equivalent, already-complete `data/nested_thomas_legacy/dtm_k{5,10,15}`
  snapshot (same param ranges/schema, an earlier draw of the same v2 generation process, not the exact
  clouds the live regeneration will produce): every finite pair there had `birth > 0` and `ratio > 1`,
  so `log(log(ratio))` was real-valued throughout, and H1 pair counts ranged from 0 to a few dozen per
  diagram -- the regime the sanity check and NaN-safe correlations above are guarding for.
- `parent_intensity`, `meta_offspring`, `meta_cluster_scale`, `mean_offspring`, `cluster_scale` are
  sampled log-uniformly and independently per parameter vector (`docs/nested_thomas_v2_data_report.md`
  Sec. 4), so a Pearson r here is a marginal association, not a controlled ablation -- the other four
  params are free to vary alongside the one being tested.
- The CSV cache (`notebooks/out/loglog_stats_nested_thomas.csv`) holds the per-`(k, diagram, dim)`
  summary stats, not the raw per-pair `log(log(ratio))` arrays -- adding a 10th stat still needs a
  fresh (`FORCE_RECOMPUTE = True`) pass over `diagrams.pkl`, the cache only speeds up replotting/adding
  correlation views against the existing nine.
